# Case 2: 2D Upper-Body Interpersonal Dynamics (MOSAIC) — reproduction

Pairs converse across four background-noise levels (Office 60 → Café 70 → Food
Court 80 → Party 87.5 dB SPL), each partner recorded by a dedicated webcam. Case 2
quantifies **interpersonal coordination** via cross-recurrence between the two
partners' ROI velocity-magnitude signals.

> **Data note.** The paper's dyadic figure needs *both* partners (the left- and
> right-camera files) across all 47 pairs. This notebook runs the **individual-level**
> pipeline (ROI velocity linear metrics only -- the paper never reports
> individual-level recurrence for Case 2) on whatever single-camera trials are
> present; the dyadic cross-recurrence entry point (`run_reproduction`) is included
> and runs once both-camera data is available.

## Configuration

In [ ]:
import glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/Volumes/X9_Pro/Case_Study_2_Mosaic"
from pose_dynamics.case_studies.mosaic import default_conditions_csv
CONDITIONS_CSV = default_conditions_csv()

files = sorted(f for f in glob.glob(f"{DATA_DIR}/*.csv") if "/._" not in f)
print(f"{len(files)} camera files present")

## 1. Load a trial and resolve ROIs

`load_mosaic_file` reads the name-based OpenPose export and keeps only the ROI
keypoints — the manuscript's BODY_25 groups (arms, upper body) plus the face
region — resolving them by name.

In [ ]:
from pose_dynamics.case_studies.mosaic import load_mosaic_file

seq, roi_map = load_mosaic_file(files[0])
print(seq.summary())
print("ROIs (keypoints each):", {roi: len(idx) for roi, idx in roi_map.items()})

## 2. ROI velocity-magnitude signals

`roi_velocity_signals` runs the individual preprocessing (mask 0.30 → interpolate →
10 Hz filter → normalize to [0,1] → downsample to 30 Hz), aggregates each ROI to
its centroid, and takes the velocity magnitude — the one-dimensional movement-
intensity signal per ROI.

In [ ]:
from pose_dynamics.case_studies.mosaic import roi_velocity_signals

feats = roi_velocity_signals(seq, roi_map)
feats.plot(names=[f"{r}_speed" for r in roi_map]);

## 3. Choosing the embedding parameters (human-in-the-loop)

At 30 Hz we estimate `(τ, m)` from AMI/FNN across the ROI signals, then commit.

In [ ]:
from pose_dynamics.embedding import select_embedding, Signal, plot_embedding_evidence

signals = []
for f in files:
    s, rm = load_mosaic_file(f)
    fs = roi_velocity_signals(s, rm)
    for roi in rm:
        signals.append(Signal(f"{Path(f).stem}_{roi}", fs.get(f"{roi}_speed"), group={"roi": roi}))

evidence = select_embedding(signals, tau_grid=(10, 20), m_grid=(3, 6),
                            ami_max_lag=40, fnn_max_dim=8, subset=40, seed=0)
plot_embedding_evidence(evidence)
print(evidence.justification)

In [ ]:
params = evidence.commit(tau=10, m=4, notes="Case 2: committed from AMI/FNN")
params.to_dict()

## 4. Individual-level analysis (runnable now)

Per ROI, per 60 s window (50% overlap): RMS/mean/SD of velocity magnitude, grouped
by background-noise condition. No recurrence analysis here -- the paper's Case 2
individual-level results are linear-metrics only; dyadic CRQA is in Section 5.

In [ ]:
from pose_dynamics.case_studies.mosaic import run_individual, plot_individual_figure

df = run_individual(files, CONDITIONS_CSV, progress=True)
print(f"{len(df)} window-rows")
df.groupby(["roi", "condition"], observed=True)[["rms", "mean_vel", "sd_vel"]].mean().round(4)

In [ ]:
plot_individual_figure(df, roi="arms");

## 5. Dyadic reproduction (needs both partners)

This is the paper's analysis — interpersonal cross-recurrence between the two
partners' ROI signals, plus zero-lag velocity cross-correlation. It runs only when
both the `left` and `right` camera files are present for a session-trial.

In [ ]:
from pose_dynamics.case_studies.mosaic import run_reproduction

try:
    dyad_df = run_reproduction(DATA_DIR, CONDITIONS_CSV, progress=True)
    dyad_df.groupby(["roi", "condition"], observed=True)[["cross_perc_recur", "cross_lmax", "xcorr_lag0"]].mean()
except FileNotFoundError as e:
    print("Dyadic reproduction unavailable:", e)

## Notes

- **Name-based ROIs.** MOSAIC's OpenPose export uses named columns; the loader
  resolves the manuscript's BODY_25 ROI groups by name (arms, upper body) and the
  face region by substring, matching the prototype's name-based sets.
- **Individual vs. dyadic.** Movement intensity (RMS velocity) rising with
  background noise is recoverable per participant; the interpersonal coordination
  finding (cross-recurrence increasing with noise) requires both partners and is
  produced by `run_reproduction` once the full dataset is available.
- **Stats** (dataset-specific, notebook-only) can be fit on the tidy tables with
  statsmodels, as in the Case 1 and Case 3 notebooks.